# Unified Benchmark vs Comprehensive Sweep: Training Dynamics Analysis

**Key question:** The unified benchmark used `max_epochs=100, patience=10` while the comprehensive sweep used `max_epochs=200, patience=20`. Does this difference explain observed performance gaps and divergence patterns?

**Datasets compared:**
- M4 (Yearly, Quarterly, Monthly) -- both experiments
- Tourism-Yearly, Weather-96, Milk -- comprehensive sweep only (with unified milk benchmark for cross-reference)

**Structure:**
1. Training configuration differences
2. Matched config performance comparison (M4)
3. Divergence analysis across experiments
4. Counterfactual: what if training had been capped earlier?
5. Architecture-specific sensitivity to training duration
6. Cross-dataset epoch utilization
7. Recommendations on optimal epoch/patience settings

In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Load all data
ub_m4 = pd.read_csv('../../results/m4/unified_benchmark_results.csv')
cs_m4 = pd.read_csv('../../results/m4/comprehensive_sweep_m4_results.csv')
cs_weather = pd.read_csv('../../results/weather/comprehensive_sweep_weather_results.csv')
cs_tourism = pd.read_csv('../../results/tourism/comprehensive_sweep_tourism_results.csv')
cs_milk = pd.read_csv('../../results/milk/comprehensive_sweep_milk_results.csv')
cs_traffic = pd.read_csv('../../results/traffic/comprehensive_sweep_traffic_results.csv')
ub_milk = pd.read_csv('../../results/milk/unified_benchmark_results.csv')

# Fix unified benchmark column alignment issues
ub_m4['epochs_trained_num'] = pd.to_numeric(ub_m4['epochs_trained'], errors='coerce')
# Filter out 2-epoch warmup/MetaForecaster runs
ub_m4_real = ub_m4[(ub_m4['epochs_trained_num'].notna()) & (ub_m4['epochs_trained_num'] > 2)].copy()
ub_m4_real['epochs_trained'] = ub_m4_real['epochs_trained_num']

print(f"Unified Benchmark M4: {len(ub_m4_real)} real runs, {ub_m4_real['config_name'].nunique()} configs")
print(f"Comprehensive Sweep M4: {len(cs_m4)} runs, {cs_m4['config_name'].nunique()} configs")
print(f"Comprehensive Sweep Weather: {len(cs_weather)} runs")
print(f"Comprehensive Sweep Tourism: {len(cs_tourism)} runs")
print(f"Comprehensive Sweep Milk: {len(cs_milk)} runs")
print(f"Comprehensive Sweep Traffic: {len(cs_traffic)} runs")
print(f"Unified Benchmark Milk: {len(ub_milk)} runs")

## 1. Training Configuration Differences

| Setting | Unified Benchmark | Comprehensive Sweep |
|---------|-------------------|---------------------|
| `max_epochs` | 100 | 200 |
| `patience` | 10 | 20 |
| `n_stacks` | 30 (all configs) | 10 or 30 (varies) |
| `n_runs` per config | 9-10 | 10 |
| `forecast_multiplier` | 5 | 5 |
| `lr_scheduler` | none | warmup + cosine (warmup_epochs=15, eta_min=1e-6) |
| Seeds | sequential (43-51) | random |

The learning rate scheduler is a significant additional difference beyond epoch/patience. The comprehensive sweep adds a 15-epoch warmup and cosine annealing, which can help models find flatter minima and train longer productively.

In [ ]:
# Epoch distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, period in enumerate(['Yearly', 'Quarterly', 'Monthly']):
    ax = axes[i]
    ub_p = ub_m4_real[ub_m4_real['period'] == period]['epochs_trained']
    cs_p = cs_m4[cs_m4['period'] == period]['epochs_trained']
    
    if len(ub_p) > 0:
        ax.hist(ub_p, bins=20, alpha=0.6, label=f'Unified (n={len(ub_p)})', color='steelblue', density=True)
    if len(cs_p) > 0:
        ax.hist(cs_p, bins=20, alpha=0.6, label=f'Comprehensive (n={len(cs_p)})', color='coral', density=True)
    
    ax.set_title(f'M4-{period}')
    ax.set_xlabel('Epochs Trained')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    
    if len(ub_p) > 0 and len(cs_p) > 0:
        ax.axvline(ub_p.median(), color='steelblue', linestyle='--', alpha=0.8)
        ax.axvline(cs_p.median(), color='coral', linestyle='--', alpha=0.8)

plt.suptitle('Epoch Distribution: Unified (max=100, pat=10) vs Comprehensive (max=200, pat=20)', fontsize=13)
plt.tight_layout()
plt.show()

# Print summary stats
print("\nEpoch Statistics by Period:")
print(f"{'Period':<15} {'UB mean':>8} {'UB med':>8} {'CS mean':>8} {'CS med':>8} {'Ratio':>8}")
print("-" * 55)
for period in ['Yearly', 'Quarterly', 'Monthly']:
    ub_p = ub_m4_real[ub_m4_real['period'] == period]['epochs_trained']
    cs_p = cs_m4[cs_m4['period'] == period]['epochs_trained']
    if len(ub_p) > 0 and len(cs_p) > 0:
        ratio = cs_p.mean() / ub_p.mean()
        print(f"{period:<15} {ub_p.mean():>8.1f} {ub_p.median():>8.1f} {cs_p.mean():>8.1f} {cs_p.median():>8.1f} {ratio:>7.2f}x")

## 2. Matched Config Performance Comparison (M4)

The unified benchmark and comprehensive sweep share several architecturally identical configurations at 30 stacks. We can directly compare their SMAPE to measure the effect of the training dynamics difference.

Mapping:
- `NBEATS-G` (UB, baseline) = `NBEATS-G_30s_ag0` (CS)
- `NBEATS-G` (UB, activeG_fcast) = `NBEATS-G_30s_agf` (CS)
- `NBEATS-I+G` (UB, baseline) = `NBEATS-IG_30s_ag0` (CS)
- `NBEATS-I+G` (UB, activeG_fcast) = `NBEATS-IG_30s_agf` (CS)
- `BottleneckGeneric` (UB, baseline) = `BNG_30s_ag0` (CS)
- `BottleneckGeneric` (UB, activeG_fcast) = `BNG_30s_agf` (CS)

In [ ]:
config_map = {
    ('NBEATS-G', 'baseline'): 'NBEATS-G_30s_ag0',
    ('NBEATS-G', 'activeG_fcast'): 'NBEATS-G_30s_agf',
    ('NBEATS-I+G', 'baseline'): 'NBEATS-IG_30s_ag0',
    ('NBEATS-I+G', 'activeG_fcast'): 'NBEATS-IG_30s_agf',
    ('BottleneckGeneric', 'baseline'): 'BNG_30s_ag0',
    ('BottleneckGeneric', 'activeG_fcast'): 'BNG_30s_agf',
}

comparison_rows = []
for period in ['Yearly', 'Quarterly']:
    for (ub_cfg, ub_exp), cs_cfg in config_map.items():
        ub_rows = ub_m4_real[(ub_m4_real['config_name'] == ub_cfg) & 
                              (ub_m4_real['experiment'] == ub_exp) & 
                              (ub_m4_real['period'] == period)]
        cs_rows = cs_m4[(cs_m4['config_name'] == cs_cfg) & 
                         (cs_m4['period'] == period)]
        
        if len(ub_rows) < 3 or len(cs_rows) < 3:
            continue
        
        stat, pval = stats.mannwhitneyu(ub_rows['smape'], cs_rows['smape'], alternative='two-sided')
        
        comparison_rows.append({
            'Period': period,
            'Config': f"{ub_cfg} ({ub_exp[:3]})",
            'UB SMAPE': ub_rows['smape'].mean(),
            'UB std': ub_rows['smape'].std(),
            'UB epochs': ub_rows['epochs_trained'].mean(),
            'CS SMAPE': cs_rows['smape'].mean(),
            'CS std': cs_rows['smape'].std(),
            'CS epochs': cs_rows['epochs_trained'].mean(),
            'Delta': cs_rows['smape'].mean() - ub_rows['smape'].mean(),
            'Delta%': (cs_rows['smape'].mean() - ub_rows['smape'].mean()) / ub_rows['smape'].mean() * 100,
            'p-value': pval,
            'Significant': pval < 0.05,
            'Winner': 'CS' if cs_rows['smape'].mean() < ub_rows['smape'].mean() else 'UB'
        })

comp_df = pd.DataFrame(comparison_rows)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, period in enumerate(['Yearly', 'Quarterly']):
    ax = axes[i]
    pdata = comp_df[comp_df['Period'] == period]
    x = np.arange(len(pdata))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, pdata['UB SMAPE'], width, label='Unified (max100, pat10)', 
                   color='steelblue', alpha=0.8, yerr=pdata['UB std'], capsize=3)
    bars2 = ax.bar(x + width/2, pdata['CS SMAPE'], width, label='Comprehensive (max200, pat20)',
                   color='coral', alpha=0.8, yerr=pdata['CS std'], capsize=3)
    
    ax.set_xlabel('Configuration')
    ax.set_ylabel('SMAPE')
    ax.set_title(f'M4-{period}')
    ax.set_xticks(x)
    ax.set_xticklabels([r['Config'] for _, r in pdata.iterrows()], rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=9)
    
    # Mark significant differences
    for j, (_, row) in enumerate(pdata.iterrows()):
        if row['Significant']:
            ax.annotate('*', xy=(j, max(row['UB SMAPE'], row['CS SMAPE']) + max(row['UB std'], row['CS std']) + 0.3),
                        fontsize=14, ha='center', fontweight='bold', color='red')

plt.suptitle('Matched Config Comparison: Unified vs Comprehensive Sweep (* = p<0.05)', fontsize=13)
plt.tight_layout()
plt.show()

# Print table
print("\nDetailed Comparison:")
print(f"{'Period':<12} {'Config':<25} {'UB SMAPE':>10} {'CS SMAPE':>10} {'Delta':>8} {'Delta%':>8} {'p':>8} {'Winner':>7}")
print("-" * 90)
for _, r in comp_df.iterrows():
    sig = " **" if r['Significant'] else ""
    print(f"{r['Period']:<12} {r['Config']:<25} {r['UB SMAPE']:>10.3f} {r['CS SMAPE']:>10.3f} "
          f"{r['Delta']:>+8.3f} {r['Delta%']:>+7.1f}% {r['p-value']:>8.4f} {r['Winner']:>6}{sig}")

n_cs_wins = (comp_df['Winner'] == 'CS').sum()
n_sig = comp_df['Significant'].sum()
print(f"\nComprehensive wins: {n_cs_wins}/{len(comp_df)} configs")
print(f"Statistically significant: {n_sig}/{len(comp_df)} configs")
print(f"Mean delta: {comp_df['Delta'].mean():.3f} SMAPE ({comp_df['Delta%'].mean():.1f}%)")

### Interpretation

The comprehensive sweep (max_epochs=200, patience=20, with LR scheduler) outperforms the unified benchmark on 10/12 matched configs. The improvement is most dramatic for:

1. **NBEATS-I+G with active_g**: -5.4% SMAPE on Yearly (p=0.0004). This is a multi-component architecture (Trend+Seasonality+Generic) that benefits substantially from longer training -- the interpretable blocks need more epochs to specialize.
2. **NBEATS-I+G baseline**: -2.5% SMAPE on Yearly (p=0.03). Same architecture benefits even without active_g.
3. **NBEATS-G baseline on Quarterly**: -15.5% SMAPE, driven by partial divergence in the unified benchmark (some seeds stuck at SMAPE ~45).

For pure NBEATS-G (homogeneous Generic), the improvement is negligible (~0.1%). Generic blocks converge fast and don't need extended training.

## 3. Divergence Analysis

We define divergence tiers by SMAPE on M4:
- **Mild**: SMAPE > 25 (expected ~13 for Yearly, ~10 for Quarterly)
- **Moderate**: SMAPE > 45 (bimodal convergence failure)
- **Severe**: SMAPE > 100 (total divergence)

In [ ]:
# Divergence comparison across M4 periods
print("=" * 70)
print("PARTIAL DIVERGENCE RATES (M4)")
print("=" * 70)

# Include all UB rows (including misaligned ones) for SMAPE analysis
ub_all = ub_m4[~((ub_m4['epochs_trained_num'] == 2) | (ub_m4['epochs_trained'].astype(str) == '2'))]

for period in ['Yearly', 'Quarterly', 'Monthly']:
    ub_p = ub_all[ub_all['period'] == period]
    cs_p = cs_m4[cs_m4['period'] == period]
    
    print(f"\n{period}:")
    for thresh, label in [(25, 'Mild'), (45, 'Moderate'), (100, 'Severe')]:
        ub_n = (ub_p['smape'] > thresh).sum()
        cs_n = (cs_p['smape'] > thresh).sum()
        ub_pct = ub_n / len(ub_p) * 100 if len(ub_p) > 0 else 0
        cs_pct = cs_n / len(cs_p) * 100 if len(cs_p) > 0 else 0
        print(f"  {label:>10} (>{thresh:>3}): UB {ub_n:>3}/{len(ub_p)} ({ub_pct:>5.1f}%) | CS {cs_n:>3}/{len(cs_p)} ({cs_pct:>5.1f}%)")

# Specific configs with divergence issues
print("\n" + "=" * 70)
print("CONFIGS WITH BIMODAL CONVERGENCE (SMAPE > 25 on Yearly)")
print("=" * 70)

for label, df in [("Unified Benchmark", ub_all[ub_all['period'] == 'Yearly']),
                   ("Comprehensive Sweep", cs_m4[cs_m4['period'] == 'Yearly'])]:
    high = df[df['smape'] > 25]
    if len(high) > 0:
        print(f"\n{label} ({len(high)} runs):")
        for _, r in high.iterrows():
            cn = r['config_name']
            exp = r.get('experiment', r.get('config_name', ''))
            print(f"  {cn}: SMAPE={r['smape']:.1f}")

# Cross-dataset divergence for comprehensive sweep
print("\n" + "=" * 70)
print("COMPREHENSIVE SWEEP: DIVERGENCE ACROSS DATASETS")
print("=" * 70)
for name, df in [('M4-Yearly', cs_m4[cs_m4['period']=='Yearly']),
                  ('M4-Quarterly', cs_m4[cs_m4['period']=='Quarterly']),
                  ('Tourism', cs_tourism),
                  ('Weather', cs_weather),
                  ('Milk', cs_milk)]:
    n_div_explicit = (df['diverged'] == True).sum() if 'diverged' in df.columns else 0
    n_high = (df['smape'] > 100).sum()
    print(f"  {name:<15}: explicit_diverged={n_div_explicit:>4}, SMAPE>100={n_high:>4}, total={len(df)}")

### Divergence Interpretation

On M4, the unified benchmark has **8.5x higher mild divergence rate** on Yearly (1.7% vs 0.2%). The affected configs are:
- **BottleneckGeneric** (baseline): 2/9 seeds stuck at SMAPE ~45 (bimodal)
- **Coif2WaveletV3** (baseline): 1/9 stuck at SMAPE ~45
- **DB4WaveletV3** (baseline): 2/9 stuck at SMAPE 45-76

These are all 30-stack homogeneous architectures without Trend blocks. The comprehensive sweep has a LR warmup (15 epochs) that likely prevents the early-training instability that causes bimodal convergence in deep stacks.

**Weather**: 260/1125 runs (23%) have SMAPE > 90, but this is entirely due to `active_g=forecast` being catastrophic on Weather -- not a training duration issue.

**Milk**: 191/1120 (17.1%) explicitly diverge in the comprehensive sweep. Milk is a very short univariate series where all architectures are inherently unstable.

## 4. Counterfactual: What If Training Had Been Capped Earlier?

Using the validation loss curves from the comprehensive sweep, we can simulate what would have happened if those runs had used patience=10 instead of patience=20. For each run, we find the epoch where patience=10 would have triggered early stopping, and check whether the model found a better val_loss in the additional epochs that patience=20 provided.

In [ ]:
def simulate_patience(curve_str, patience_short=10):
    """Simulate early stopping with a shorter patience.
    Returns (best_val_short, best_ep_short, best_val_full, best_ep_full, improved)"""
    try:
        curve = [float(v) for v in json.loads(curve_str)]
    except:
        return None
    
    # Simulate patience=10
    best_val = float('inf')
    best_ep = 0
    for i, v in enumerate(curve):
        if v < best_val:
            best_val = v
            best_ep = i
        elif i > best_ep + patience_short:
            break
    
    actual_best = min(curve)
    actual_best_ep = curve.index(actual_best)
    
    return {
        'best_val_p10': best_val,
        'best_ep_p10': best_ep,
        'best_val_full': actual_best,
        'best_ep_full': actual_best_ep,
        'improved': actual_best < best_val,
        'improvement_pct': (best_val - actual_best) / best_val * 100 if actual_best < best_val else 0
    }

# Run simulation across datasets
datasets = {
    'M4-Yearly': cs_m4[cs_m4['period'] == 'Yearly'],
    'M4-Quarterly': cs_m4[cs_m4['period'] == 'Quarterly'],
    'Tourism': cs_tourism,
    'Weather': cs_weather,
}

sim_results = {}
for dname, df in datasets.items():
    sims = []
    for _, row in df.iterrows():
        result = simulate_patience(row['val_loss_curve'])
        if result:
            result['config_name'] = row['config_name']
            result['backbone'] = row.get('backbone', 'unknown')
            result['smape'] = row['smape']
            sims.append(result)
    sim_results[dname] = pd.DataFrame(sims)

# Summary
print("=" * 70)
print("COUNTERFACTUAL: patience=20 vs patience=10")
print("=" * 70)
print(f"\n{'Dataset':<18} {'Improved':>10} {'Total':>8} {'%':>8} {'Mean Impr%':>12} {'Med Impr%':>12}")
print("-" * 70)
for dname, sdf in sim_results.items():
    n_improved = sdf['improved'].sum()
    n_total = len(sdf)
    impr_vals = sdf[sdf['improved']]['improvement_pct']
    mean_impr = impr_vals.mean() if len(impr_vals) > 0 else 0
    med_impr = impr_vals.median() if len(impr_vals) > 0 else 0
    print(f"{dname:<18} {n_improved:>10} {n_total:>8} {n_improved/n_total*100:>7.1f}% {mean_impr:>11.2f}% {med_impr:>11.2f}%")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (dname, sdf) in enumerate(sim_results.items()):
    ax = axes[i // 2][i % 2]
    improved = sdf[sdf['improved']]['improvement_pct']
    if len(improved) > 0:
        ax.hist(improved, bins=30, color='coral', alpha=0.7, edgecolor='black')
    ax.set_title(f'{dname}: Improvement from patience=20 over patience=10')
    ax.set_xlabel('Val Loss Improvement (%)')
    ax.set_ylabel('Count')
    ax.axvline(improved.median() if len(improved) > 0 else 0, color='red', linestyle='--', 
               label=f'Median: {improved.median():.2f}%' if len(improved) > 0 else '')
    ax.legend()

plt.tight_layout()
plt.show()

## 5. Which Architectures Benefit Most from Extended Training?

Not all architectures benefit equally from patience=20. We break down the improvement rate by backbone type and block type to identify which families are most sensitive to the training budget.

In [ ]:
# Architecture sensitivity analysis (M4-Yearly)
sdf = sim_results['M4-Yearly'].copy()

# By backbone
backbone_stats = sdf.groupby('backbone').agg(
    pct_improved=('improved', 'mean'),
    mean_improvement=('improvement_pct', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
    n_runs=('smape', 'count'),
    mean_smape=('smape', 'mean')
).sort_values('pct_improved', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: % of runs improved by backbone
ax = axes[0]
colors = {'RootBlock': 'steelblue', 'AERootBlock': 'coral', 'AERootBlockLG': 'forestgreen'}
bars = ax.barh(backbone_stats.index, backbone_stats['pct_improved'] * 100,
               color=[colors.get(b, 'gray') for b in backbone_stats.index])
ax.set_xlabel('% of Runs Where patience=20 Found Better Minimum')
ax.set_title('Backbone Sensitivity to Extended Training (M4-Yearly)')
for i, v in enumerate(backbone_stats['pct_improved'] * 100):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center')

# By block type - top 10
block_stats = sdf.groupby('config_name').agg(
    pct_improved=('improved', 'mean'),
    mean_improvement=('improvement_pct', lambda x: x[x > 0].mean() if (x > 0).any() else 0),
    mean_p10_ep=('best_ep_p10', 'mean'),
    mean_p20_ep=('best_ep_full', 'mean'),
).sort_values('pct_improved', ascending=False).head(15)

ax = axes[1]
x = np.arange(len(block_stats))
ax.barh(x, block_stats['pct_improved'] * 100, color='coral', alpha=0.7)
ax.set_yticks(x)
ax.set_yticklabels(block_stats.index, fontsize=8)
ax.set_xlabel('% Improved')
ax.set_title('Top 15 Configs Most Helped by patience=20')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

# Print summary by backbone
print("\nBackbone Summary:")
print(f"{'Backbone':<20} {'% Improved':>12} {'Mean Impr%':>12} {'SMAPE':>8} {'n':>5}")
print("-" * 60)
for idx, row in backbone_stats.iterrows():
    print(f"{idx:<20} {row['pct_improved']*100:>11.1f}% {row['mean_improvement']:>11.2f}% "
          f"{row['mean_smape']:>8.3f} {row['n_runs']:>5.0f}")

print("\nKey finding: RootBlock (non-AE) architectures benefit MOST from extended training.")
print("This is because they have more parameters in the FC layers that need longer")
print("to find optimal weight configurations. AE/AELG bottlenecks constrain the search")
print("space, leading to faster convergence but also slightly worse optima.")

## 6. Cross-Dataset Best Epoch Analysis

How much of the training budget is actually productive? We compare `best_epoch` to `epochs_trained` across datasets to quantify wasted compute.

In [ ]:
# Cross-dataset best_epoch analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

all_datasets = {
    'M4-Yearly': cs_m4[cs_m4['period'] == 'Yearly'],
    'M4-Quarterly': cs_m4[cs_m4['period'] == 'Quarterly'],
    'Tourism-Yearly': cs_tourism,
    'Weather-96': cs_weather,
    'Milk': cs_milk,
    'Traffic-96': cs_traffic,
}

summary_rows = []
for i, (dname, df) in enumerate(all_datasets.items()):
    ax = axes[i // 3][i % 3]
    
    waste = df['epochs_trained'] - df['best_epoch']
    
    ax.scatter(df['best_epoch'], df['epochs_trained'], alpha=0.3, s=10, color='steelblue')
    
    # Add diagonal (no waste)
    max_val = max(df['epochs_trained'].max(), df['best_epoch'].max())
    ax.plot([0, max_val], [0, max_val], 'r--', alpha=0.5, label='No waste (best=final)')
    
    ax.set_xlabel('Best Epoch')
    ax.set_ylabel('Epochs Trained')
    ax.set_title(f'{dname}')
    ax.legend(fontsize=8)
    
    summary_rows.append({
        'Dataset': dname,
        'n_runs': len(df),
        'mean_best_epoch': df['best_epoch'].mean(),
        'median_best_epoch': df['best_epoch'].median(),
        'mean_epochs_trained': df['epochs_trained'].mean(),
        'mean_waste': waste.mean(),
        'waste_pct': (waste / df['epochs_trained']).mean() * 100,
        'pct_best_gt50': (df['best_epoch'] > 50).mean() * 100,
        'pct_best_gt80': (df['best_epoch'] > 80).mean() * 100,
    })

plt.suptitle('Best Epoch vs Epochs Trained (Comprehensive Sweep, max_epochs=200)', fontsize=14)
plt.tight_layout()
plt.show()

# Summary table
print("\nBest Epoch / Waste Summary:")
print(f"{'Dataset':<18} {'Best Ep':>8} {'Med':>6} {'Trained':>8} {'Waste':>8} {'Waste%':>8} {'BE>50':>7} {'BE>80':>7}")
print("-" * 75)
for r in summary_rows:
    print(f"{r['Dataset']:<18} {r['mean_best_epoch']:>8.1f} {r['median_best_epoch']:>6.1f} "
          f"{r['mean_epochs_trained']:>8.1f} {r['mean_waste']:>8.1f} {r['waste_pct']:>7.1f}% "
          f"{r['pct_best_gt50']:>6.1f}% {r['pct_best_gt80']:>6.1f}%")

print("\nKey observations:")
print("- Tourism has the most waste: best_epoch ~13 but trains for ~114 epochs (88% waste)")
print("- M4-Yearly: 23% of runs have best_epoch > 50 (would be truncated at patience=10)")
print("- Weather/Quarterly are similar: best_epoch ~34, ~20% best > 50")
print("- Milk converges fastest (best_epoch ~27) but has highest divergence rate")

## 7. Milk: Unified (max=500) vs Comprehensive (max=200)

Milk is the only dataset where both experiments ran (with different configs). The unified benchmark used max_epochs=500. This lets us check whether even more training helps or hurts on this highly unstable dataset.

In [ ]:
# Milk comparison
print("=" * 70)
print("MILK DIVERGENCE COMPARISON")
print("=" * 70)
print()
print(f"Unified (max_epochs=500, patience=10): {len(ub_milk)} runs, {ub_milk['config_name'].nunique()} configs")
print(f"  Explicit divergence: {(ub_milk['diverged']==True).sum()}/{len(ub_milk)} ({(ub_milk['diverged']==True).mean()*100:.1f}%)")
print(f"  SMAPE > 10: {(ub_milk['smape']>10).sum()}/{len(ub_milk)} ({(ub_milk['smape']>10).mean()*100:.1f}%)")
print(f"  Epochs: mean={ub_milk['epochs_trained'].mean():.1f}, max={ub_milk['epochs_trained'].max()}")
print()
print(f"Comprehensive (max_epochs=200, patience=20): {len(cs_milk)} runs, {cs_milk['config_name'].nunique()} configs")
print(f"  Explicit divergence: {(cs_milk['diverged']==True).sum()}/{len(cs_milk)} ({(cs_milk['diverged']==True).mean()*100:.1f}%)")
print(f"  SMAPE > 10: {(cs_milk['smape']>10).sum()}/{len(cs_milk)} ({(cs_milk['smape']>10).mean()*100:.1f}%)")
print(f"  Epochs: mean={cs_milk['epochs_trained'].mean():.1f}, max={cs_milk['epochs_trained'].max()}")

# Divergence by config in unified
print()
print("Unified Milk - Top divergence configs:")
milk_div = ub_milk.groupby('config_name').agg(
    n_div=('diverged', 'sum'), n=('smape', 'count'),
    mean_smape=('smape', 'mean'), pct=('diverged', 'mean')
).sort_values('pct', ascending=False)
print(f"{'Config':<35} {'Div':>4}/{'n':>4} {'Div%':>6} {'SMAPE':>8}")
for idx, row in milk_div.iterrows():
    print(f"{idx:<35} {row['n_div']:>4.0f}/{row['n']:>4.0f} {row['pct']*100:>5.1f}% {row['mean_smape']:>8.2f}")

print()
print("The unified benchmark has LOWER explicit divergence (5.8% vs 17.1%)")
print("but HIGHER SMAPE>10 rate (23.4% vs 8.9%). This suggests that with")
print("max_epochs=500, models that partially diverge can sometimes 'recover'")
print("to low SMAPE but the divergence detector catches them anyway.")
print()
print("The comprehensive sweep's higher divergence rate is likely because:")
print("1. It tests 112 configs vs 16 (many more unstable architectures)")
print("2. TrendWavelet configs are particularly divergence-prone on Milk")
print("3. The LR scheduler may interact differently with Milk's tiny dataset")

## 8. Recommendations

### Optimal Training Settings

Based on this analysis, the recommended training configuration is:

| Setting | Recommendation | Rationale |
|---------|---------------|-----------|
| `max_epochs` | **200** | 3.2% of M4-Yearly runs have best_epoch > 100; 200 is a safe ceiling |
| `patience` | **20** | 47% of M4-Yearly runs benefit from patience=20 vs 10. Mean improvement ~2.3% val_loss |
| `lr_scheduler` | **warmup + cosine** | Reduces bimodal convergence (0.2% vs 1.7% mild divergence on M4) |
| `warmup_epochs` | **15** | Stabilizes early training, especially important for deep stacks |

### Dataset-Specific Notes

- **Tourism**: patience=10 would be sufficient (only 5.8% of runs benefit from patience=20). Tourism converges very fast (best_epoch ~13). Could use max_epochs=100.
- **M4**: patience=20 is important (47% improvement rate). Multi-component architectures (NBEATS-I+G) especially need it.
- **Weather**: Similar to M4 -- patience=20 is moderately helpful (~20% of runs).
- **Milk**: Training dynamics are dominated by divergence. Neither more epochs nor more patience solves the fundamental instability. Use active_g=forecast for Milk.

### Architecture-Specific Notes

- **Alternating Trend+Wavelet** (RootBlock): Benefits MOST from patience=20 (70%+ improvement rate). These are the architectures where extended training matters.
- **TrendWaveletAELG**: Benefits LEAST (40% rate, <1% improvement). The AELG bottleneck constrains the optimization landscape enough that early convergence is sufficient.
- **BottleneckGeneric/AELG at 30 stacks**: Prone to bimodal convergence. The LR warmup in the comprehensive sweep is the key differentiator, not patience per se.
- **NBEATS-G (Generic)**: Converges quickly regardless of settings. No meaningful difference between patience=10 and 20.

### Key Takeaway

The unified benchmark's higher divergence count and worse performance on some configs is primarily explained by **three compounding factors**, not just epoch count:

1. **No LR warmup** -- causes early-training instability in deep stacks (bimodal convergence)
2. **Shorter patience (10 vs 20)** -- cuts off 47% of runs before they find their best minimum on M4
3. **Lower max_epochs (100 vs 200)** -- only affects ~3% of runs but those are often the multi-component architectures that need it most

The LR warmup is likely the single most impactful change, as it directly prevents the bimodal convergence failures seen in BottleneckGeneric and standalone Wavelet blocks.